# Streaming Responses: SSE, Async, Concurrency

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/03-streaming-sse-async.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.

> **Note:** This lesson builds a service that must run as a long-lived process. The cells below show the full implementation but cannot run end-to-end in Colab. To run this lesson: clone the repo locally and use `uvicorn main:app` or `docker compose up`.


In [ ]:
!pip install -q anthropic fastapi pydantic uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

Your `/generate` endpoint works but users are unhappy. A typical Claude response takes 8-12 seconds to complete. The user clicks the button, sees nothing for 10 seconds, then the entire response appears at once. They think the page is broken. Many abandon before the response arrives.

The model is not slow -- it produces tokens continuously from the first millisecond. The problem is you are buffering all of them before sending anything. The fix is streaming: send each token to the client as soon as the model produces it. The user sees the first word in under a second and watches the response build in real time.

Streaming also solves a timeout problem. A 10-second HTTP request is close to ma...

## The Concept

### Buffered vs. Streaming Response

```
BUFFERED (current behavior):
User clicks        API call starts        Model finishes      User sees output
    |                    |                     |                    |
    |<---- 10 seconds ----------------------------------->|
    |                                          buffer fills -> response sent


STREAMING (what we want):
User clicks   API call starts   Token 1   Token 2   ...   Done
    |               |              |         |              |
    |               |<-- ~0.3s -->|<~0.05s>|<~0.05s>...   |
    |                          ...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 06-03: Streaming Responses -- SSE, Async, Concurrency

A FastAPI service demonstrating Server-Sent Events (SSE) streaming of
Anthropic model output. Includes:
- AsyncAnthropic client (required for non-blocking streaming)
- Async generator producing SSE-formatted events
- StreamingResponse with correct headers
- A minimal HTML client served at /
- Concurrent stream handling via asyncio

Usage:
    pip install fastapi uvicorn anthropic pydantic
    export ANTHROPIC_API_KEY=sk-ant-...
    uvicorn main:app --reload --port 8000

Test with curl:
    curl -X POST http://localhost:8000/stream --no-buffer \
        -H "Content-Type: application/json" \
        -d '{"prompt": "Count from 1 to 10."}'

Test in browser:
    open http://localhost:8000
"""

import json
import logging
import os
from contextlib import asynccontextmanager
from datetime import datetime, timezone

import anthropic
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import HTMLResponse, StreamingResponse
from pydantic import BaseModel, Field

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
)
log = logging.getLogger(__name__)

### Request model

In [ ]:
class StreamRequest(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=4000)
    max_tokens: int = Field(default=1024, ge=1, le=4096)
    system: str | None = Field(default=None, max_length=2000)

### Lifespan: initialize AsyncAnthropic client once at startup

In [ ]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    """Initialize the async Anthropic client at startup."""
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "ANTHROPIC_API_KEY is not set. "
            "Export it before starting: export ANTHROPIC_API_KEY=sk-ant-..."
        )

    model = os.environ.get("MODEL", "claude-3-5-haiku-20241022")
    timeout = float(os.environ.get("TIMEOUT_SECONDS", "60.0"))

    # AsyncAnthropic is required for streaming in async FastAPI routes.
    # The synchronous Anthropic client blocks the event loop and prevents
    # concurrent requests from being served.
    app.state.client = anthropic.AsyncAnthropic(
        api_key=api_key,
        timeout=timeout,
    )
    app.state.model = model

    log.info("Startup complete: model=%s timeout=%.1fs", model, timeout)

    yield

    log.info("Shutdown")


app = FastAPI(
    title="Streaming AI Service",
    description="Server-Sent Events streaming of Anthropic model output",
    version="1.0.0",
    lifespan=lifespan,
)

### Async generator: produces SSE-formatted events

In [ ]:
async def stream_tokens(
    client: anthropic.AsyncAnthropic,
    model: str,
    prompt: str,
    system: str | None = None,
    max_tokens: int = 1024,
):
    """
    Async generator that streams model output as SSE events.

    SSE format:
        data: {"token": "Hello"}\n\n
        data: {"token": " world"}\n\n
        data: {"done": true, "input_tokens": 10, "output_tokens": 5}\n\n

    Each yield is one SSE event. FastAPI's StreamingResponse sends each event
    to the client as soon as it is yielded -- no buffering.

    The async context manager keeps the connection open for the duration of
    the stream. The event loop can switch to other requests between yields.
    """
    kwargs: dict = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    if system:
        kwargs["system"] = system

    try:
        async with client.messages.stream(**kwargs) as stream:
            async for text in stream.text_stream:
                # Each token is a separate SSE event.
                # json.dumps handles escaping of special characters.
                event_data = json.dumps({"token": text})
                yield f"data: {event_data}\n\n"

            # After all tokens, emit a final event with usage stats.
            # This is the signal the client uses to detect clean stream completion
            # vs. a dropped connection.
            final = await stream.get_final_message()
            done_data = json.dumps({
                "done": True,
                "input_tokens": final.usage.input_tokens,
                "output_tokens": final.usage.output_tokens,
                "model": final.model,
            })
            yield f"data: {done_data}\n\n"

    except anthropic.APIStatusError as e:
        log.error("Anthropic API error in stream: status=%d", e.status_code)
        error_data = json.dumps({"error": f"API error: HTTP {e.status_code}"})
        yield f"data: {error_data}\n\n"
    except anthropic.APITimeoutError:
        log.error("Anthropic API timeout in stream")
        error_data = json.dumps({"error": "Request timed out"})
        yield f"data: {error_data}\n\n"
    except Exception as e:
        log.error("Unexpected error in stream: %s", e, exc_info=True)
        error_data = json.dumps({"error": "Internal server error"})
        yield f"data: {error_data}\n\n"

### Routes

In [ ]:
@app.get("/health", tags=["ops"])
async def health(req: Request):
    """Health check for load balancers. Fast and cheap -- no model call."""
    return {
        "status": "ok",
        "model": req.app.state.model,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }


@app.post("/stream", tags=["model"])
async def stream_endpoint(req: Request, body: StreamRequest):
    """
    Stream model output as Server-Sent Events.

    The response is a continuous text/event-stream body.
    Each event is a JSON object on a 'data:' line followed by two newlines.

    Headers:
    - Content-Type: text/event-stream
    - Cache-Control: no-cache (required for SSE)
    - X-Accel-Buffering: no (disables nginx response buffering)

    Use curl --no-buffer or EventSource in the browser to consume this endpoint.
    """
    client: anthropic.AsyncAnthropic = req.app.state.client
    model: str = req.app.state.model

    log.info(
        "POST /stream model=%s prompt_chars=%d max_tokens=%d",
        model,
        len(body.prompt),
        body.max_tokens,
    )

    return StreamingResponse(
        stream_tokens(
            client=client,
            model=model,
            prompt=body.prompt,
            system=body.system,
            max_tokens=body.max_tokens,
        ),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "X-Accel-Buffering": "no",  # prevents nginx from buffering the stream
        },
    )


@app.get("/", response_class=HTMLResponse, tags=["demo"])
async def demo_page():
    """
    Minimal HTML page that demonstrates consuming the /stream endpoint.
    Open http://localhost:8000 in a browser to try streaming interactively.
    """
    return """<!DOCTYPE html>
<html>
<head>
    <title>Streaming Demo</title>
    <style>
        body { font-family: monospace; max-width: 800px; margin: 40px auto; padding: 0 20px; }
        #output { white-space: pre-wrap; border: 1px solid #ccc; padding: 16px; min-height: 100px; }
        #stats { color: #666; font-size: 0.9em; margin-top: 8px; }
        button { padding: 8px 16px; margin: 8px 4px; cursor: pointer; }
        textarea { width: 100%; height: 80px; font-family: monospace; }
    </style>
</head>
<body>
    <h1>Streaming Demo</h1>
    <textarea id="prompt">Tell me a short story about a robot who learns to paint.</textarea>
    <br>
    <button onclick="startStream()">Stream</button>
    <button onclick="clearOutput()">Clear</button>
    <div id="output">Output will appear here...</div>
    <div id="stats"></div>

    <script>
    async function startStream() {
        const prompt = document.getElementById('prompt').value;
        const output = document.getElementById('output');
        const stats = document.getElementById('stats');
        output.textContent = '';
        stats.textContent = 'Streaming...';

        const startTime = Date.now();
        let firstTokenTime = null;
        let tokenCount = 0;

        const response = await fetch('/stream', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({prompt: prompt})
        });

        const reader = response.body.getReader();
        const decoder = new TextDecoder();
        let buffer = '';

        while (true) {
            const {done, value} = await reader.read();
            if (done) break;

            buffer += decoder.decode(value, {stream: true});
            const events = buffer.split('\\n\\n');
            buffer = events.pop();  // keep incomplete event in buffer

            for (const event of events) {
                if (!event.startsWith('data: ')) continue;
                const data = JSON.parse(event.slice(6));

                if (data.token !== undefined) {
                    if (firstTokenTime === null) {
                        firstTokenTime = Date.now() - startTime;
                    }
                    output.textContent += data.token;
                    tokenCount++;
                }

                if (data.done) {
                    const totalTime = ((Date.now() - startTime) / 1000).toFixed(1);
                    stats.textContent = [
                        'Done.',
                        `First token: ${firstTokenTime}ms`,
                        `Total: ${totalTime}s`,
                        `Input tokens: ${data.input_tokens}`,
                        `Output tokens: ${data.output_tokens}`
                    ].join(' | ');
                }

                if (data.error) {
                    output.textContent += `\\n[Error: ${data.error}]`;
                    stats.textContent = 'Stream ended with error.';
                }
            }
        }
    }

    function clearOutput() {
        document.getElementById('output').textContent = '';
        document.getElementById('stats').textContent = '';
    }
    </script>
</body>
</html>"""


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

### Demo

In [ ]:
import uvicorn

uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=True)

---

## Self-Check

Answer these without running code first:

**1. What is the root cause?**

- A. The synchronous Anthropic client is not thread-safe and causes data corruption
- B. The synchronous client blocks the asyncio event loop during the API call, preventing other requests from being handled until the current stream completes
- C. SSE does not support concurrent connections and queues requests automatically
- D. FastAPI limits streaming endpoints to 10 concurrent connections by default

**2. What is the most likely cause of the browser receiving batched tokens?**

- A. Browsers buffer SSE events and only display them after 500ms by default
- B. An nginx reverse proxy is buffering the streaming response; add X-Accel-Buffering: no to the response headers and proxy_buffering off to the nginx config
- C. The StreamingResponse media_type should be application/json, not text/event-stream
- D. EventSource in the browser only supports GET requests, not POST

**3. What is the correct server-side design to enable the client to distinguish a complete stream from a dropped connection?**

- A. Set the HTTP status code to 206 Partial Content for incomplete streams
- B. Emit a final event with a 'done' field (e.g., data: {"done": true}) after all tokens, so the client can verify the stream completed normally vs. the connection dropping mid-stream
- C. Use WebSockets instead of SSE, which have built-in close codes
- D. Send a Content-Length header with the expected number of tokens before streaming begins

**4. Which protocol is most appropriate, and why?**

- A. SSE, because it handles all three requirements natively
- B. WebSockets, because requirement (1) and (3) involve client-to-server communication that SSE cannot handle -- SSE is unidirectional (server to client only)
- C. SSE, because WebSockets do not support JSON payloads
- D. WebSockets are never appropriate for AI applications because they add too much latency

**5. What happens to the client, and what is the correct fix?**

- A. FastAPI automatically sends a 500 status code to replace the 200
- B. The connection closes abruptly with no signal; the client sees a truncated stream identical to a dropped connection. Fix: catch exceptions inside the generator and yield an error event before the generator exits
- C. The generator retries from the beginning automatically
- D. The client receives a JSON error body appended to the token stream

**6. How do you explain the value of streaming to the product manager?**

- A. Streaming makes the model produce tokens faster by reducing internal buffering
- B. Streaming does not change total generation time, but reduces perceived wait time from 8.2s (nothing visible) to 0.4s (first content visible), which dramatically reduces user abandonment and perceived performance
- C. Streaming reduces API costs by sending fewer tokens per request
- D. Streaming is required for the service to handle more than one user at a time

**7. What is wrong with the SSE format?**

- A. The data field must be named 'event' not 'data'
- B. Each SSE event must be followed by two newlines (\n\n) not one; a single \n continues the same event rather than terminating it
- C. SSE events must be Base64-encoded before sending
- D. The Content-Type must be application/event-stream not text/event-stream

**8. Why does asyncio concurrency maintain low TTFT under this streaming load?**

- A. Each concurrent stream runs on a separate CPU core
- B. The service is not actually handling all 50 requests concurrently; they are queued
- C. The async event loop multiplexes all 50 streams on a single thread by suspending at each await (each token), so new requests can start between token emissions without waiting for any stream to finish
- D. FastAPI automatically distributes streaming requests across multiple processes

_Answers are in `checks.json` in the lesson directory._